# SHA-512 Padding

In [1]:
# Numerical arrays.
import numpy as np

# Memory profiling.
import sys

## Generators

In [2]:
# A list.
squares_list = [i**2 for i in range(1, 1000001)]

In [3]:
# A lot of memory.
sys.getsizeof(squares_list)

8448728

In [4]:
# A generator.
squares_gen = (i**2 for i in range(1, 1000001))

In [5]:
# Not a lot of memory.
sys.getsizeof(squares_gen)

200

In [6]:
%%timeit
# Quick.
squares_list = [i**2 for i in range(1, 1000001)]
sum(squares_list)

27 ms ± 271 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [7]:
%%timeit
# Almost as quick.
squares_list = (i**2 for i in range(1, 1000001))
sum(squares_list)

27.7 ms ± 486 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [8]:
# Generator functions.
def squares_gen_func(n):
    """Generate squares of numbers from 1 to n."""
    for i in range(1, n + 1):
        yield i**2

In [9]:
for i in squares_gen_func(10):
    print(i)

1
4
9
16
25
36
49
64
81
100


## Reading Bytes

In [10]:
# Reading strings.
with open("padding.ipynb") as f:
    one = f.read(1)
    print(one, type(one))

{ <class 'str'>


In [11]:
# Reading bytes.
with open("padding.ipynb", "rb") as f:
    one = f.read(1)
    print(one, type(one))

b'{' <class 'bytes'>


## Reading NumPy Arrays

In [12]:
# Reading bytes.
with open("padding.ipynb", "rb") as f:
    block = f.read(128)

# Show.
print(block)

b'{\n "cells": [\n  {\n   "cell_type": "markdown",\n   "id": "4bd45473",\n   "metadata": {},\n   "source": [\n    "# SHA-512 Padding"\n   '


In [13]:
# To a numpy array, big-endian uint64.
block_np = np.frombuffer(block, dtype=">u8")

# Show.
print(block_np)

[8865934148500810860 8296257374210629664 8865934138782409573
 7812724407439222050 4188385506636229476 8032009553485373472
 2315529037213736994 3774689613083588403 2462354229359026797
 7310575178888798498 4188483430994550816 2315540080970851173
 2466319321322758176 2315451780165026093 3832899872237904996
 7597123017317031968]


## Generate Blocks

In [14]:
def blocks(filename):
    """Generate SHA-512 message blocks as arrays of 16 uint64 values."""

    # Open the file in binary mode.
    with open(filename, "rb") as f:
        # Keep track of the total length of the file in bytes.
        length = 0
        # Read the first 128-byte block.
        block = f.read(128)

        # While we get full blocks...
        while len(block) == 128:
            # Increase length counter.
            length += 128
            # Yield the block as a numpy array of 16 uint64 values.
            yield np.frombuffer(block, dtype=">u8").astype(np.uint64)
            # Read the next 128-byte block.
            block = f.read(128)

        # Increase length counter for the final, possibly partial block.
        length += len(block)

        # Pad the final, possibly empty block.
        # Append a single 1 bit (0x80) to the end of the block. We have to use bytes.
        padding = block + b"\x80"
        # Append enough 0 bytes (0x00) to leave just enough room for the 16-byte length at the end.
        padding += bytes((112 - len(padding)) % 128)
        # The length of the message in bits is appended as a 128-bit big-endian integer.
        padding += (length * 8).to_bytes(16, "big")

        # Yield the padded blocks as numpy arrays of 16 uint64 values.
        # There can be one or two padded blocks, depending on the length of the original message.
        for i in range(0, len(padding), 128):
            yield np.frombuffer(padding[i:i + 128], dtype=">u8").astype(np.uint64)

In [15]:
# Example usage of the blocks generator function.
for block in blocks("padding.ipynb"):
    print(block)

[8865934148500810860 8296257374210629664 8865934138782409573
 7812724407439222050 4188385506636229476 8032009553485373472
 2315529037213736994 3774689613083588403 2462354229359026797
 7310575178888798498 4188483430994550816 2315540080970851173
 2466319321322758176 2315451780165026093 3832899872237904996
 7597123017317031968]
[6704206319205485088 2340475564443312739 7308335452547280997
 2466319078007989349 2462354229359026789 8675449613314518894
 6873460006452470330 2319683705072787490 7594232504696060217
 3834591015929782828  729618462022657396 7017841271823809056
 8898316860754239522 8031453536840872738 4188448109183508512
 2315540080970851173]
[2466319321322758176 2315451780084100453 8244229862955442546
 8242002328670727714 3173384209785954921 7885925484890975861
 7885936127506587758 8096467323152113696 2314887988848503818
 2314885530852204621 7308620307741306994 8027219300180715310
 6660298471471325216 2315529077180559988 2338346166934708256
 2332031315148090378]
[231498549188730064

## End